# Function Vectors

A function vector is a single d_model vector extractable from a few-shot prompt by reading a specific attention head's output. It causally encodes an abstract task and can transfer that task to new contexts.

**References:**
- Todd et al. (2023) "Function Vectors in Large Language Models"
- Hendel et al. (2023) "In-Context Learning Creates Task Vectors"

## Why This Matters

Function vectors capture the representation of a task (like 'capitalize' or 'translate to French') as a direction in activation space. These vectors can transfer tasks between contexts, providing evidence that models represent tasks and inputs in separable subspaces.

**Key references:**
- [Zou et al. (2023) "Representation Engineering"](https://arxiv.org/abs/2310.01405) — Reading and controlling model behavior via representations

In [ ]:
import jax
import jax.numpy as jnp
import numpy as np

from irtk import HookedTransformer, HookedTransformerConfig
from irtk.function_vectors import (
    extract_function_vector,
    scan_for_function_heads,
    function_vector_transfer,
    function_vector_arithmetic,
    function_vector_similarity_matrix,
)

# Create a small model
cfg = HookedTransformerConfig(
    n_layers=4, d_model=32, n_ctx=64, d_head=8, n_heads=4, d_vocab=100,
)
model = HookedTransformer(cfg)
key = jax.random.PRNGKey(42)
leaves, treedef = jax.tree.flatten(model)
new_leaves = []
for leaf in leaves:
    if isinstance(leaf, jnp.ndarray) and leaf.dtype == jnp.float32:
        key, subkey = jax.random.split(key)
        new_leaves.append(jax.random.normal(subkey, leaf.shape) * 0.1)
    else:
        new_leaves.append(leaf)
model = jax.tree.unflatten(treedef, new_leaves)

few_shot = jnp.array([0, 5, 10, 15, 20, 25, 30, 35])
zero_shot = jnp.array([40, 45, 50, 55, 60, 65, 70, 75])
print(f"Model: {cfg.n_layers} layers, {cfg.n_heads} heads")

## 1. Extract Function Vector

Read the output of a specific attention head at the final query position in a few-shot prompt to extract the function vector that encodes the task.

In [ ]:
fv_result = extract_function_vector(model, few_shot, layer=1, head=0)

print(f"Function vector shape: {fv_result['function_vector'].shape}")
print(f"FV norm: {fv_result['fv_norm']:.4f}")
print(f"Head output norm: {fv_result['head_output_norm']:.4f}")
print(f"Extraction position: {fv_result['extraction_position']}")

## 2. Scan for Function Heads

Identify which heads encode function vectors by patching each head's output from the few-shot run into a zero-shot context and measuring metric improvement.

In [ ]:
def metric(logits):
    return float(logits[-1, 0])

scan = scan_for_function_heads(model, few_shot, zero_shot, metric, top_k=5)

print(f"Baseline metric: {scan['baseline_metric']:.4f}")
print(f"\nTop function heads (by transfer effect):")
for layer, head, score in scan['top_function_heads']:
    print(f"  L{layer}H{head}: transfer_score={score:.4f}")

## 3. Function Vector Transfer

Inject a previously extracted function vector into a new (zero-shot) context to test causal sufficiency.

In [ ]:
fv = fv_result['function_vector']
transfer = function_vector_transfer(model, fv, zero_shot, inject_layer=1)

print(f"Top promoted tokens after injection:")
for tok_idx, logit_increase in transfer['top_promoted']:
    print(f"  Token {tok_idx}: +{logit_increase:.4f}")
print(f"\nMax logit diff: {float(np.max(np.abs(transfer['logit_diff']))):.4f}")

## 4. Function Vector Arithmetic

Test whether function vectors compose linearly: does fv_a + fv_b produce the combined effect?

In [ ]:
fv_a = extract_function_vector(model, few_shot, layer=0, head=0)['function_vector']
fv_b = extract_function_vector(model, few_shot, layer=1, head=1)['function_vector']

arith = function_vector_arithmetic(model, fv_a, fv_b, zero_shot, inject_layer=0)

print(f"Linearity score: {arith['linearity_score']:.4f}")
print(f"  (1.0 = perfectly linear composition)")
print(f"Cosine similarity between fv_a and fv_b: {arith['cosine_similarity']:.4f}")

## 5. Function Vector Similarity Matrix

Extract function vectors for multiple tasks and compare their geometry. Similar tasks should have similar function vectors.

In [ ]:
tasks = [
    jnp.array([0, 1, 2, 3, 4, 5, 6, 7]),
    jnp.array([10, 11, 12, 13, 14, 15, 16, 17]),
    jnp.array([20, 21, 22, 23, 24, 25, 26, 27]),
]

sim = function_vector_similarity_matrix(model, tasks, layer=1, head=0)

print(f"Similarity matrix:")
for i in range(3):
    row = [f"{sim['similarity_matrix'][i, j]:.3f}" for j in range(3)]
    print(f"  Task {i}: {row}")

print(f"\nMean off-diagonal similarity: {sim['mean_similarity']:.4f}")
print(f"FV norms: {[f'{n:.3f}' for n in sim['norms']]}")